## Evaluation on ground truth

In [6]:
import joblib
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score
from sentence_transformers import SentenceTransformer
import pandas as pd
import spacy

In [7]:
ground_truth = pd.read_csv(r'..\data\ground_truth.csv')
ground_truth.head()

,review_title,hotel_name,avg_rating,nationality,rating,review_text,year,month,season,residence_type,trip_type,traveller_type,stay_length,label
0,Very poor,Van der Valk Hotel Antwerpen,8.4,Belgium,1.0,There are no comments available for this review,2021,April,Spring,suite,leisure trip,couple,1,0
1,Pleasant with friendly staff.,Novotel Brussels City Centre,8.5,Netherlands,10.0,The ONLY place in front of the hotel to drop o...,2019,March,Spring,double,leisure trip,group,1,1
2,Good,Hostel Bruegel,8.4,Brazil,7.5,"Nothing. Everything was good. ,\n\nIs cheap an...",2019,July,Summer,double,leisure trip,people with friends,1,1
3,Free shuttlebus from the airport,Thon Hotel Brussels Airport,7.7,Luxembourg,7.9,10 min drive from the airpory with free shuttl...,2019,May,Spring,double,leisure trip,couple,1,1
4,Very good,B-aparthotel Ambiorix,7.7,United Kingdom,8.0,The air on/heating we couldn’t work it out mad...,2019,December,Winter,apartment,leisure trip,group,2,1


In [8]:
ground_truth.info()

<class 'pandas.DataFrame'>
RangeIndex: 2051 entries, 0 to 2050
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   review_title    2051 non-null   str    
 1   hotel_name      2051 non-null   str    
 2   avg_rating      2051 non-null   float64
 3   nationality     2051 non-null   str    
 4   rating          2051 non-null   float64
 5   review_text     2051 non-null   str    
 6   year            2051 non-null   int64  
 7   month           2051 non-null   str    
 8   season          2051 non-null   str    
 9   residence_type  2051 non-null   str    
 10  trip_type       2051 non-null   str    
 11  traveller_type  2051 non-null   str    
 12  stay_length     2051 non-null   int64  
 13  label           2051 non-null   int64  
dtypes: float64(2), int64(3), str(9)
memory usage: 224.5 KB


In [9]:
ground_truth['review'] = ground_truth['review_title'] + ' ' + ground_truth['review_text']

In [10]:
nlp = spacy.load('en_core_web_sm')

def preprocess(text):
    doc = nlp(text)
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
    ]
    return ' '.join(tokens)

In [11]:
ground_truth['clean_text'] = ground_truth['review'].apply(preprocess)

In [12]:

X_gt = ground_truth['clean_text']
y_gt = ground_truth['label']

In [13]:
# ── 1. LR + TF-IDF Bigrams ───────────────────────────────────────────────────
model_lr_tfidf = joblib.load(r'..\models\model_lr_tfidf.pkl')
vec_lr_tfidf   = joblib.load(r'..\models\vectorizer_lr_tfidf.pkl')

X_gt_tfidf     = vec_lr_tfidf.transform(X_gt)
y_pred_tfidf   = model_lr_tfidf.predict(X_gt_tfidf)

print("=== LR + TF-IDF Bigrams - Ground Truth Evaluation ===")
print(classification_report(y_gt, y_pred_tfidf))

# ── 2. LR + MiniLM ───────────────────────────────────────────────────────────
model_lr_minilm = joblib.load(r'..\models\model_lr_minilm.pkl')
st_model        = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Encoding ground truth with MiniLM...")
X_gt_bert     = st_model.encode(X_gt.tolist(), batch_size=32, show_progress_bar=True)
y_pred_minilm = model_lr_minilm.predict(X_gt_bert)

print("=== LR + MiniLM - Ground Truth Evaluation ===")
print(classification_report(y_gt, y_pred_minilm))

# ── Comparison Table ──────────────────────────────────────────────────────────
def get_metrics(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True)
    return {
        'Accuracy':        accuracy_score(y_true, y_pred),
        'Precision (Neg)': report['0']['precision'],
        'Recall (Neg)':    report['0']['recall'],
        'F1 (Neg)':        report['0']['f1-score'],
    }

gt_comparison = pd.DataFrame([
    {'Model': 'LR + TF-IDF Bigrams', **get_metrics(y_gt, y_pred_tfidf)},
    {'Model': 'LR + MiniLM',         **get_metrics(y_gt, y_pred_minilm)},
]).round(3)

gt_comparison

=== LR + TF-IDF Bigrams - Ground Truth Evaluation ===
              precision    recall  f1-score   support

           0       0.34      0.90      0.49       291
           1       0.98      0.71      0.82      1760

    accuracy                           0.74      2051
   macro avg       0.66      0.81      0.66      2051
weighted avg       0.89      0.74      0.78      2051



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7685.02it/s]


Encoding ground truth with MiniLM...


Batches: 100%|██████████| 65/65 [00:04<00:00, 15.48it/s]

=== LR + MiniLM - Ground Truth Evaluation ===
              precision    recall  f1-score   support

           0       0.41      0.85      0.55       291
           1       0.97      0.80      0.87      1760

    accuracy                           0.80      2051
   macro avg       0.69      0.82      0.71      2051
weighted avg       0.89      0.80      0.83      2051



,Model,Accuracy,Precision (Neg),Recall (Neg),F1 (Neg)
0,LR + TF-IDF Bigrams,0.738,0.340,0.900,0.494
1,LR + MiniLM,0.803,0.407,0.845,0.549
